# PetroMind RAG 

In [2]:
from config import (
    MODEL_NAME, EMBEDDING_DIM, PG_CONN_STR, WO_TABLE,
    TOP_K, CHUNK_SIZE_TOKENS, CHUNK_OVERLAP_TOKENS,
    DATA_DIR, HDF5_FILES,
)
from embedding import load_embedder, encode_text
from kaggle_downloader import download_single_file, download_dataset, list_local_files
from read_ncmapss import extract_engine_events
from chunking import split_text_to_chunks_tokens, add_embeddings_to_chunks
from db import get_connection, create_chunks_table, drop_table, insert_chunks_batch, retrieve_top_k
from llm import get_llm_client, topk_to_string, generate_text
import os
import h5py
import numpy as np

my_embedder = load_embedder(MODEL_NAME)
print(f'Embedding dim: {my_embedder.get_embedding_dimension()}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2769.72it/s]


Embedding dim: 384


In [3]:
list_local_files(DATA_DIR)


Local HDF5 files in 'E:\petromind\data':
  [OK]  N-CMAPSS_DS01-005.h5  (2.87 GB)
  [--]  N-CMAPSS_DS02-006.h5  (not downloaded)
  [--]  N-CMAPSS_DS03-012.h5  (not downloaded)
  [--]  N-CMAPSS_DS04.h5  (not downloaded)
  [--]  N-CMAPSS_DS05.h5  (not downloaded)
  [--]  N-CMAPSS_DS06.h5  (not downloaded)
  [--]  N-CMAPSS_DS07.h5  (not downloaded)
  [--]  N-CMAPSS_DS08a-009.h5  (not downloaded)
  [--]  N-CMAPSS_DS08c-008.h5  (not downloaded)
  [--]  N-CMAPSS_DS08d-010.h5  (not downloaded)
  Total: 2.87 GB



In [4]:
import shutil
import kagglehub

os.environ['KAGGLE_CACHE_DIR'] = r'E:\petromind\kaggle_cache'

downloaded_path = kagglehub.dataset_download(
    'bishals098/nasa-cmapss-2-engine-degradation',
    path='N-CMAPSS_DS01-005.h5'
)
print('Downloaded to:', downloaded_path)

os.makedirs(DATA_DIR, exist_ok=True)
dst = os.path.join(DATA_DIR, 'N-CMAPSS_DS01-005.h5')
if not os.path.exists(dst):
    shutil.copy2(downloaded_path, dst)
    print('Copied to:', dst)
else:
    print('File already exists in data dir.')

list_local_files(DATA_DIR)

Downloaded to: C:\Users\LAPTOP\.cache\kagglehub\datasets\bishals098\nasa-cmapss-2-engine-degradation\versions\1\N-CMAPSS_DS01-005.h5
File already exists in data dir.

Local HDF5 files in 'E:\petromind\data':
  [OK]  N-CMAPSS_DS01-005.h5  (2.87 GB)
  [--]  N-CMAPSS_DS02-006.h5  (not downloaded)
  [--]  N-CMAPSS_DS03-012.h5  (not downloaded)
  [--]  N-CMAPSS_DS04.h5  (not downloaded)
  [--]  N-CMAPSS_DS05.h5  (not downloaded)
  [--]  N-CMAPSS_DS06.h5  (not downloaded)
  [--]  N-CMAPSS_DS07.h5  (not downloaded)
  [--]  N-CMAPSS_DS08a-009.h5  (not downloaded)
  [--]  N-CMAPSS_DS08c-008.h5  (not downloaded)
  [--]  N-CMAPSS_DS08d-010.h5  (not downloaded)
  Total: 2.87 GB



In [5]:
H5_PATH = os.path.join(DATA_DIR, 'N-CMAPSS_DS01-005.h5')

with h5py.File(H5_PATH, 'r') as h5:
    A_dev  = h5['A_dev'][:]
    A_test = h5['A_test'][:]

dev_units  = np.unique(A_dev[:, 0]).astype(int).tolist()
test_units = np.unique(A_test[:, 0]).astype(int).tolist()

print(f'Dev  unit IDs : {dev_units}')
print(f'Test unit IDs : {test_units}')
print()
print('Compare with config.py UNIT_FAILURE_MODE keys:')
from config import UNIT_FAILURE_MODE
for uid in dev_units + test_units:
    mode = UNIT_FAILURE_MODE.get(uid, 'UNKNOWN — needs fixing!')
    print(f'  unit {uid:>3} -> {mode}')

Dev  unit IDs : [1, 2, 3, 4, 5, 6]
Test unit IDs : [7, 8, 9, 10]

Compare with config.py UNIT_FAILURE_MODE keys:
  unit   1 -> LPT_efficiency_flow_HPT_combined
  unit   2 -> HPT_efficiency_degradation
  unit   3 -> LPT_efficiency_flow_HPT_combined
  unit   4 -> LPT_efficiency_flow_HPT_combined
  unit   5 -> HPT_efficiency_degradation
  unit   6 -> LPT_efficiency_flow_HPT_combined
  unit   7 -> HPT_LPT_complex
  unit   8 -> HPT_LPT_complex
  unit   9 -> HPT_LPT_complex
  unit  10 -> HPT_LPT_complex


In [6]:
# dev=[1,2,3,4,5,6]  test=[7,8,9,10]
# Units 2,5 = HPT    Units 1,3,4,6 = LPT combined
# Test units 7,8,9,10 = HPT_LPT_complex

from config import UNIT_FAILURE_MODE

UNIT_FAILURE_MODE[1]  = 'LPT_efficiency_flow_HPT_combined'
UNIT_FAILURE_MODE[2]  = 'HPT_efficiency_degradation'
UNIT_FAILURE_MODE[3]  = 'LPT_efficiency_flow_HPT_combined'
UNIT_FAILURE_MODE[4]  = 'LPT_efficiency_flow_HPT_combined'
UNIT_FAILURE_MODE[5]  = 'HPT_efficiency_degradation'
UNIT_FAILURE_MODE[6]  = 'LPT_efficiency_flow_HPT_combined'
UNIT_FAILURE_MODE[7]  = 'HPT_LPT_complex'
UNIT_FAILURE_MODE[8]  = 'HPT_LPT_complex'
UNIT_FAILURE_MODE[9]  = 'HPT_LPT_complex'
UNIT_FAILURE_MODE[10] = 'HPT_LPT_complex'

print('Fixed UNIT_FAILURE_MODE:')
for k, v in sorted(UNIT_FAILURE_MODE.items()):
    print(f'  unit {k:>3} -> {v}')

Fixed UNIT_FAILURE_MODE:
  unit   1 -> LPT_efficiency_flow_HPT_combined
  unit   2 -> HPT_efficiency_degradation
  unit   3 -> LPT_efficiency_flow_HPT_combined
  unit   4 -> LPT_efficiency_flow_HPT_combined
  unit   5 -> HPT_efficiency_degradation
  unit   6 -> LPT_efficiency_flow_HPT_combined
  unit   7 -> HPT_LPT_complex
  unit   8 -> HPT_LPT_complex
  unit   9 -> HPT_LPT_complex
  unit  10 -> HPT_LPT_complex


In [7]:
res = extract_engine_events(H5_PATH, split='dev')
print(f'\nTotal event records: {len(res)}')

from collections import Counter
modes = Counter(r['failure_mode'] for r in res)
print('\nFailure mode counts:')
for mode, count in modes.items():
    print(f'  {mode}: {count}')


[read_ncmapss] N-CMAPSS_DS01-005.h5  split='dev'  sample_every=1
  W_dev            shape=(4906636, 4)  mem~157MB
  X_s_dev          shape=(4906636, 14)  mem~550MB
  X_v_dev          shape=(4906636, 14)  mem~550MB
  T_dev            shape=(4906636, 10)  mem~393MB
  Unit IDs in file: [1, 2, 3, 4, 5, 6]
  unit   1  total_cycles= 100  kept=100  mode=LPT_efficiency_flow_HPT_combined
  unit   2  total_cycles=  75  kept= 75  mode=HPT_efficiency_degradation
  unit   3  total_cycles= 100  kept=100  mode=LPT_efficiency_flow_HPT_combined
  unit   4  total_cycles=  95  kept= 95  mode=LPT_efficiency_flow_HPT_combined
  unit   5  total_cycles=  89  kept= 89  mode=HPT_efficiency_degradation
  unit   6  total_cycles=  94  kept= 94  mode=LPT_efficiency_flow_HPT_combined
[read_ncmapss] Done — 553 records from N-CMAPSS_DS01-005.h5


Total event records: 553

Failure mode counts:
  LPT_efficiency_flow_HPT_combined: 389
  HPT_efficiency_degradation: 164


In [8]:
print(res[0]['text'])

SENSOR LOG ENTRY
unit_id      : 1
flight_cycle : 1
rul          : 99.0 cycles remaining
zone         : degrading
failure_mode : LPT efficiency flow HPT combined
n_samples    : 4498 (1 Hz readings)

SCENARIO
--------
alt=8757.6ft  Mach=0.5  TRA=53.4%  T2=509.6R

PHYSICAL SENSORS X_s (14 vars)
------------------------------------------
  Wf             mean=585.5812pps  std=15.4181  max=618.8168
  Nf             mean=1355.4337rpm  std=66.6899  max=1471.4504
  Nc             mean=1650.0921rpm  std=125.0547  max=1852.8438
  T24            mean=1157.1543R  std=50.7830  max=1269.9479
  T30            mean=15.6274R  std=1.0985  max=19.4352
  T48            mean=12.3818R  std=0.7556  max=14.4882
  T50            mean=15.8654R  std=1.1152  max=19.7311
  P15            mean=19.0410psia  std=1.7388  max=24.4311
  P21            mean=276.8425psia  std=49.0815  max=394.9561
  P24            mean=281.9742psia  std=49.5042  max=401.4552
  Ps30           mean=12.4526psia  std=1.0923  max=15.9748
  P40

In [9]:
chunks = split_text_to_chunks_tokens(
    res,
    embedder=my_embedder,
    chunk_size=CHUNK_SIZE_TOKENS,
    overlap=CHUNK_OVERLAP_TOKENS,
)
print(f'Total chunks: {len(chunks)}')

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1196 > 256). Running this sequence through the model will result in indexing errors


Total chunks: 2765


In [10]:
enriched_chunks = add_embeddings_to_chunks(chunks, my_embedder)
print(f'Enriched chunks : {len(enriched_chunks)}')
print(f'Embedding shape : {enriched_chunks[0]["embedding"].shape}')

Enriched chunks : 2765
Embedding shape : (384,)


In [11]:
conn = get_connection(PG_CONN_STR)
print('Connected:', conn)

Connected: <psycopg.Connection [IDLE] (host=localhost port=5433 user=postgres database=petromind) at 0x215f4517050>


In [12]:
drop_table(conn, WO_TABLE)
create_chunks_table(conn, WO_TABLE, EMBEDDING_DIM)

[db] Table 'work_orders_rag' dropped.
[db] Table 'work_orders_rag' ready (dim=384).


In [13]:
insert_chunks_batch(conn, WO_TABLE, enriched_chunks)
print('\nAll data saved to PostgreSQL!')

[db] Inserted 2765 chunks into 'work_orders_rag'.

All data saved to PostgreSQL!


In [14]:
with conn.cursor() as cur:
    cur.execute(f'SELECT COUNT(*) FROM {WO_TABLE}')
    total = cur.fetchone()[0]

    cur.execute(f'SELECT failure_mode, COUNT(*) FROM {WO_TABLE} GROUP BY failure_mode ORDER BY COUNT(*) DESC')
    by_mode = cur.fetchall()

    cur.execute(f'SELECT zone, COUNT(*) FROM {WO_TABLE} GROUP BY zone ORDER BY COUNT(*) DESC')
    by_zone = cur.fetchall()

    cur.execute(f'SELECT unit_id, failure_mode, COUNT(*) FROM {WO_TABLE} GROUP BY unit_id, failure_mode ORDER BY unit_id')
    by_unit = cur.fetchall()

print(f'Total chunks in DB: {total}')
print('\nBy failure mode:')
for row in by_mode:
    print(f'  {row[0]}: {row[1]}')
print('\nBy zone:')
for row in by_zone:
    print(f'  {row[0]}: {row[1]}')
print('\nBy unit:')
for row in by_unit:
    print(f'  unit {row[0]:>3}  {row[1]:<40}  chunks={row[2]}')

Total chunks in DB: 2765

By failure mode:
  LPT_efficiency_flow_HPT_combined: 1945
  HPT_efficiency_degradation: 820

By zone:
  critical (alert zone): 1530
  degrading: 1235

By unit:
  unit   1  LPT_efficiency_flow_HPT_combined          chunks=500
  unit   2  HPT_efficiency_degradation                chunks=375
  unit   3  LPT_efficiency_flow_HPT_combined          chunks=500
  unit   4  LPT_efficiency_flow_HPT_combined          chunks=475
  unit   5  HPT_efficiency_degradation                chunks=445
  unit   6  LPT_efficiency_flow_HPT_combined          chunks=470


---
## USER SIDE — Query → Retrieve → Answer

In [15]:
userQ = 'What are the main sensor deviations before HPT efficiency failure?'

# userQ = 'Which health parameters show degradation in the critical zone?'
# userQ = 'What is the typical RUL when entering the alert zone?'
# userQ = 'What happens to fuel flow and fan speed near engine failure?'
# userQ = 'What is the difference between healthy and critical zone sensor readings?'
# userQ = 'What health parameter delta values indicate HPT blade wear?'

userQ_emb = encode_text(my_embedder, userQ)
print(f'Query: {userQ}')
print(f'Embedding shape: {userQ_emb.shape}')

Query: What are the main sensor deviations before HPT efficiency failure?
Embedding shape: (384,)


In [16]:
top5 = retrieve_top_k(conn, WO_TABLE, userQ_emb, k=TOP_K)
print(f'Retrieved {len(top5)} chunks')
for i, r in enumerate(top5):
    print(f'  [{i+1}] unit={r["unit_id"]}  cycle={r["cycle_id"]}  '
          f'rul={r["rul"]:.0f}  zone={r["zone"]}  score={r["score"]}')

Retrieved 5 chunks
  [1] unit=4  cycle=24  rul=71  zone=degrading  score=0.6391
  [2] unit=4  cycle=19  rul=76  zone=degrading  score=0.6384
  [3] unit=3  cycle=23  rul=77  zone=degrading  score=0.6383
  [4] unit=3  cycle=24  rul=76  zone=degrading  score=0.6374
  [5] unit=4  cycle=42  rul=53  zone=degrading  score=0.6372


In [17]:
context_str = topk_to_string(top5)
print(context_str)

Unit ID      : 4
Cycle        : 24
RUL          : 71.0 cycles remaining
Zone         : degrading
Failure mode : LPT_efficiency_flow_HPT_combined
Score        : 0.6391
Content      :
sensor log entry = = = = = = = = = = = = = = = = unit _ id : 4 flight _ cycle : 24 rul : 71. 0 cycles remaining zone : degrading failure _ mode : lpt efficiency flow hpt combined n _ samples : 3750 ( 1 hz readings ) scenario - - - - - - - - alt = 7366. 3ft mach = 0. 4 tra = 49. 4 % t2 = 510. 8r physical sensors x _ s ( 14 vars ) - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - wf mean = 583. 2090pps std = 19. 1339 max = 623. 5461 nf mean = 1343. 9559rpm std = 81. 2232 max = 1489. 8393 nc mean = 1627. 4881rpm std = 151. 3258 max = 1887. 1477 t24 mean = 1155. 7933r std = 63. 9468 max = 1287. 2036 t30 mean = 15. 8833r std = 1. 4039 max = 19. 6570 t48 mean = 12. 7119r std = 0. 8719 max = 14. 4277 t50 mean

---

Unit ID      : 4
Cycle        : 19
RUL          : 76.0 cycles rema

In [18]:
llm_client = get_llm_client()
answer = generate_text(context_str, userQ, client=llm_client)
print(answer)

Answer:  
- Unit ID 4, Cycle 24: T48 mean = 12.7119R (vs. Cycle 19: 11.9450R), T50 mean not fully reported — but trend shows rising downstream temperatures.  
- Unit ID 4, Cycle 42 (later cycle, RUL 53.0): T48 mean = 12.1437R, T50 mean = 15R (explicitly reported), indicating continued T48–T50 rise as RUL decreases from 71.0 (Cycle 24) to 53.0 (Cycle 42).  
- Unit ID 3, Cycle 24: T48 mean = 9.3145R, T50 mean not reported — but T48 dropped significantly vs. Cycle 23 (10.6405R), likely due to lower alt/mach/TRA (alt = 15877.6ft, mach = 0.4, tra = 56.2% vs. Cycle 23: alt = 14075.8ft, mach = 0.5, tra = 58.8%).  
- NF (fan speed) mean decreased in Unit ID 3, Cycle 24 (1295.3748rpm) vs. Cycle 23 (1337.8000rpm), while NC (core speed) dropped more sharply (1585.6583rpm vs. 1639.6832rpm), suggesting core performance degradation.  
- WF (fuel flow) mean decreased in Unit ID 3, Cycle 24 (554.6166pps) vs. Cycle 23 (574.1285pps), consistent with reduced thrust demand or efficiency loss.  

Source:  